# Baseline models

This notebook evaluates the benchmark forecasting models on the
cleaned train, validation and test splits. Simple statistical models are also fit directly here.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current available evaluation metrics include:

1. **Cumulative log-change errors and directional accuracy**
2. **MASE and persistence-relative error measures**
3. **Pearson correlation in cumulative log-change space**
4. **Temporal Pearson correlation between predicted and realised price levels**
5. **Temporal Pearson correlation between log returns formed from each horizon-aligned forecast series**
6. **Cross-sectional Pearson IC and Spearman Rank IC**
7. **Movement-magnitude diagnostics**

The two secondary temporal correlations are computed separately through time for each asset and then averaged over valid assets. The forecast-series log-return metric differences adjacent horizon-aligned forecasts only within the same trading session; with the current 15-minute window stride, those returns are 15-minute changes in each horizon-specific forecast series.

Bootstrap evaluation metrics over the test datset are available and used by default.

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.
6. **ModernTCN** - multiple ablations have been trained in Colab. The
    best version checkpoint (based on validation loss) is called and used to 
    predict. It is the version which takes all OHLCV as input, adds a time
    of day temporal feature and flattens series into batch (so we dont mix
    across asset - huge reduction in parameter count - 121,138 params in total
    including 256 params added for the temporal feature).


In [1]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.models.modern_tcn import ModernTCNBaseline
from src.evaluation.dynamic_graph_evaluation import (
    load_evaluation_artifacts,
)
from src.utils.config import load_yaml
from src.utils.metric_tables import (
    DEFAULT_SUMMARY_METRICS,
    make_evaluation_table,
    make_baseline_summary_table,
)
from src.visualization.forecast_plots import plot_forecast_comparison


Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

BASELINE_CACHE_ROOT = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation"
)

## Load the data and clean

In [3]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

#Set global Bootstrap params
BOOTSTRAP_N = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_SEED = 42


train samples: 167
val samples: 20
test samples: 62
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15
input features: ['open', 'high', 'low', 'close', 'volume']
targets: ['close']


## Summary metrics and forecast plots

Run the individual model sections first so that each `{model}_metric_table` exists, then rerun the summary cell below. The summary contains ordinary full-test-set values. Bootstrap confidence intervals remain available in each detailed model table.

In [13]:
models_to_display = [
    "persistence",
    "arima",
    "var",
    "garch",
    "modern_tcn",
    "kronos",
    "final_model",
]

metrics_to_display=(
    "cumulative_log_change_mae",
    #"relative_mae_vs_persistence",
    "persistence_win_rate",
    "cumulative_log_change_directional_accuracy",
    "cumulative_log_change_pearson_correlation",
    #"raw_price_temporal_pearson_correlation",
    "forecast_series_log_return_temporal_pearson_correlation",
    #"cumulative_log_change_cross_sectional_pearson_ic",
    #"cumulative_log_change_cross_sectional_spearman_rank_ic",
    #"cumulative_log_change_movement_magnitude_ratio",
    #"cumulative_log_change_temporal_absolute_correlation",
)

baseline_summary_table = make_baseline_summary_table(
    models_to_display=models_to_display,
    namespace=globals(),
    channel="close",
    metrics_to_display=metrics_to_display,
    model_display_names={"modern_tcn": "ModernTCN ","final_model": "GraphTCN",},)

percentage_columns = {
    "Win Rate",
    "Sign Acc.",
    "Pearson",
    "Price Pearson",
    "Series-Return Pearson",
    "MMR",
    "AbsRet Corr"
}
baseline_summary_formatters = {column: ("{:.5%}" if column in percentage_columns else "{:.7g}") for column in baseline_summary_table.columns}
display(baseline_summary_table.style.format(baseline_summary_formatters,na_rep="—",).set_caption("Frozen Test-Set Results"))

Use the function below to plot the forecasts of any model vs the true price. Can compare multiple models.

In [ ]:
fig, axes, selection = plot_forecast_comparison(
    models=["persistence","arima", "modern_tcn", "kronos", "final_model"],
    namespace=globals(),
    day=None,
    asset="NVDA",
    horizons=[1],
)

print(selection)

## Persistence

In [4]:
RUN = False

persistence_prediction_path = (
    BASELINE_CACHE_ROOT
    / "persistence"
    / "prediction_result.pt"
)

if RUN:
    persistence = PersistenceBaseline.from_config(
        config
    )

    persistence.fit(
        train_split=train,
        val_split=val,
    )

    persistence_result = persistence.predict(
        split=test,
        batch_size=256,
    )

    persistence_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        persistence_result,
        persistence_prediction_path,
    )

else:
    persistence_result = torch.load(
        persistence_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=train,
)


# Persistence predicts zero cumulative log change at every horizon,
# so its cumulative-log-change Pearson correlation or IC is undefined.
# The secondary price-series correlations remain defined and are retained.
persistence_undefined_metrics = {
    "cumulative_log_change_pearson_correlation",
    "cumulative_log_change_cross_sectional_pearson_ic",
    "cumulative_log_change_cross_sectional_spearman_rank_ic",
    "cumulative_log_change_temporal_absolute_correlation"

}

persistence_metric_names = [
    metric_name
    for metric_name in persistence_evaluator.available_metrics
    if metric_name not in persistence_undefined_metrics
]


persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_metric_names,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)


for metric_name in persistence_metric_names:
    metric_format = (
                "{:.2%}"
                if metric_name
                in ["cumulative_log_change_directional_accuracy",
                    "cumulative_log_change_pearson_correlation",
                    "raw_price_temporal_pearson_correlation",
                    "forecast_series_log_return_temporal_pearson_correlation",
                    "persistence_win_rate",
                    "cumulative_log_change_movement_magnitude_ratio",
                    ]
                else "{:.3g}"
            )
    metric_display = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000367,0.000352,0.000383,0.000367,7.92e-06
5,close,0.000785,0.000757,0.000815,0.000785,1.49e-05
15,close,0.00132,0.00128,0.00137,0.00132,2.45e-05
30,close,0.00184,0.00177,0.00191,0.00184,3.79e-05
60,close,0.00255,0.00243,0.0027,0.00255,6.94e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000255,—,—,—,—
5,close,0.000575,—,—,—,—
15,close,0.000977,—,—,—,—
30,close,0.00136,—,—,—,—
60,close,0.0019,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0011,—,—,—,—
5,close,0.00221,—,—,—,—
15,close,0.0037,—,—,—,—
30,close,0.00512,—,—,—,—
60,close,0.0071,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.80%,99.85%,99.83%,0.01%
60,close,99.68%,99.60%,99.72%,99.67%,0.03%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,10.66%,10.12%,11.18%,10.66%,0.27%
5,close,3.31%,3.13%,3.49%,3.31%,0.09%
15,close,1.92%,1.81%,2.05%,1.92%,0.06%
30,close,1.38%,1.29%,1.48%,1.38%,0.05%
60,close,0.94%,0.87%,1.01%,0.94%,0.04%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00%,0.00%,0.00%,0.00%,0.00%
5,close,0.00%,0.00%,0.00%,0.00%,0.00%
15,close,0.00%,0.00%,0.00%,0.00%,0.00%
30,close,0.00%,0.00%,0.00%,0.00%,0.00%
60,close,0.00%,0.00%,0.00%,0.00%,0.00%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.95,0.911,0.991,0.95,0.0205
5,close,2.04,1.97,2.12,2.04,0.0387
15,close,3.43,3.31,3.56,3.43,0.0638
30,close,4.78,4.59,4.98,4.78,0.0993
60,close,6.65,6.32,7.02,6.65,0.178


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1,1,1,1,0
5,close,1,1,1,1,0
15,close,1,1,1,1,0
30,close,1,1,1,1,0
60,close,1,1,1,1,0


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,50.00%,50.00%,50.00%,50.00%,0.00%
5,close,50.00%,50.00%,50.00%,50.00%,0.00%
15,close,50.00%,50.00%,50.00%,50.00%,0.00%
30,close,50.00%,50.00%,50.00%,50.00%,0.00%
60,close,50.00%,50.00%,50.00%,50.00%,0.00%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,91.88%,91.46%,92.25%,91.85%,0.20%
5,close,64.52%,63.21%,65.74%,64.47%,0.65%
15,close,-0.33%,-2.55%,1.70%,-0.42%,1.09%
30,close,-1.25%,-3.42%,0.63%,-1.38%,1.04%
60,close,0.13%,-2.13%,2.25%,0.04%,1.11%


## Mean

In [5]:
RUN = False

mean_prediction_path = (
    BASELINE_CACHE_ROOT
    / "mean"
    / "prediction_result.pt"
)

if RUN:
    mean = MeanBaseline.from_config(
        config
    )

    mean.fit(
        train_split=train,
        val_split=val,
    )

    mean_result = mean.predict(
        split=test,
        batch_size=256,
    )

    mean_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        mean_result,
        mean_prediction_path,
    )

else:
    mean_result = torch.load(
        mean_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_format = (
                "{:.2%}"
                if metric_name
                in ["cumulative_log_change_directional_accuracy",
                    "cumulative_log_change_pearson_correlation",
                    "raw_price_temporal_pearson_correlation",
                    "forecast_series_log_return_temporal_pearson_correlation",
                    "persistence_win_rate",
                    "cumulative_log_change_movement_magnitude_ratio",
                    ]
                else "{:.3g}"
            )
    metric_display = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00165,0.00158,0.00173,0.00165,3.69e-05
5,close,0.00179,0.00171,0.00187,0.00179,4e-05
15,close,0.00207,0.00198,0.00217,0.00208,4.84e-05
30,close,0.00243,0.00232,0.00255,0.00243,6.02e-05
60,close,0.00301,0.00285,0.0032,0.00301,8.87e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0012,—,—,—,—
5,close,0.0013,—,—,—,—
15,close,0.00151,—,—,—,—
30,close,0.00177,—,—,—,—
60,close,0.00221,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00469,—,—,—,—
5,close,0.00509,—,—,—,—
15,close,0.00588,—,—,—,—
30,close,0.00686,—,—,—,—
60,close,0.00838,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.14%,-2.72%,3.24%,0.16%,1.54%
5,close,0.91%,-0.78%,2.72%,0.92%,0.89%
15,close,0.06%,-2.66%,2.67%,0.10%,1.37%
30,close,0.09%,-3.58%,3.52%,0.15%,1.83%
60,close,-0.60%,-6.48%,4.30%,-0.47%,2.83%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.86%,99.83%,99.87%,99.85%,0.01%
5,close,99.84%,99.80%,99.85%,99.83%,0.01%
15,close,99.78%,99.73%,99.80%,99.77%,0.02%
30,close,99.70%,99.64%,99.73%,99.69%,0.02%
60,close,99.54%,99.43%,99.60%,99.53%,0.04%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,4.48,4.35,4.62,4.48,0.0692
5,close,2.23,2.19,2.29,2.24,0.0257
15,close,1.55,1.51,1.59,1.55,0.0204
30,close,1.31,1.29,1.34,1.31,0.0144
60,close,1.17,1.15,1.19,1.17,0.0108


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,13.62%,13.22%,14.05%,13.61%,0.21%
5,close,25.07%,24.49%,25.64%,25.06%,0.29%
15,close,33.45%,32.78%,34.13%,33.45%,0.35%
30,close,37.72%,37.04%,38.37%,37.71%,0.33%
60,close,40.86%,40.14%,41.57%,40.85%,0.37%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.022,0.00799,0.0359,0.022,0.00711
5,close,0.0256,0.012,0.039,0.0255,0.00691
15,close,0.0268,0.0114,0.0418,0.0267,0.00783
30,close,0.0313,0.0119,0.0499,0.0312,0.00969
60,close,0.0313,0.00946,0.052,0.0312,0.0108


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0242,0.0134,0.0352,0.0241,0.00559
5,close,0.0257,0.0131,0.0379,0.0256,0.00631
15,close,0.0307,0.0166,0.0444,0.0305,0.00712
30,close,0.0358,0.0188,0.0522,0.0357,0.00853
60,close,0.0301,0.011,0.0477,0.0299,0.0093


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,22.01%,19.84%,24.00%,21.84%,1.06%
5,close,11.31%,8.88%,13.34%,11.06%,1.14%
15,close,-0.41%,-2.83%,1.81%,-0.64%,1.18%
30,close,-0.42%,-2.73%,1.73%,-0.60%,1.15%
60,close,2.66%,-0.64%,6.19%,2.50%,1.77%


## ARIMA

In [15]:
RUN = False

arima_prediction_path = (
    BASELINE_CACHE_ROOT
    / "arima"
    / "prediction_result.pt"
)

if RUN:
    arima = ArimaBaseline.from_config(
        config,
        fit_mode="simple",
        optim_method="powell",
    )

    arima.fit(
        train_split=train,
        val_split=val,
    )

    arima_result = arima.predict(
        split=test,
        batch_size=32,
    )

    arima_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        arima_result,
        arima_prediction_path,
    )

else:
    arima_result = torch.load(
        arima_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_format = (
                "{:.2%}"
                if metric_name
                in ["cumulative_log_change_directional_accuracy",
                    "raw_price_temporal_pearson_correlation",
                    "forecast_series_log_return_temporal_pearson_correlation",
                    "persistence_win_rate",
                    "cumulative_log_change_movement_magnitude_ratio",
                    ]
                else "{:.3g}"
            )
    metric_display = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000367,0.000352,0.000383,0.000367,7.9e-06
5,close,0.000785,0.000757,0.000815,0.000785,1.49e-05
15,close,0.00132,0.00128,0.00137,0.00132,2.45e-05
30,close,0.00184,0.00177,0.00192,0.00184,3.8e-05
60,close,0.00256,0.00243,0.0027,0.00256,6.98e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000255,—,—,—,—
5,close,0.000575,—,—,—,—
15,close,0.000978,—,—,—,—
30,close,0.00136,—,—,—,—
60,close,0.0019,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00109,—,—,—,—
5,close,0.0022,—,—,—,—
15,close,0.0037,—,—,—,—
30,close,0.00512,—,—,—,—
60,close,0.0071,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0317,0.00286,0.0557,0.0318,0.0135
5,close,0.016,1.22e-05,0.0323,0.0159,0.00815
15,close,-0.00129,-0.0148,0.0115,-0.00124,0.00672
30,close,0.000689,-0.0137,0.0152,0.000709,0.00745
60,close,-0.000526,-0.022,0.0202,-0.000568,0.0108


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.80%,99.85%,99.83%,0.01%
60,close,99.68%,99.60%,99.72%,99.67%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1,1,1.01,1,0.000979
5,close,0.999,0.998,1,0.999,0.000426
15,close,1,1,1,1,0.000286
30,close,1,1,1,1,0.000372
60,close,1,1,1,1,0.000607


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,45.84%,45.31%,46.39%,45.84%,0.28%
5,close,48.63%,48.05%,49.21%,48.63%,0.29%
15,close,48.96%,48.35%,49.58%,48.97%,0.31%
30,close,48.97%,48.12%,49.81%,48.98%,0.43%
60,close,48.51%,47.27%,49.71%,48.51%,0.62%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.05,0.0419,0.0582,0.05,0.00421
5,close,0.02,0.00989,0.0298,0.02,0.00508
15,close,0.0093,-0.000641,0.0191,0.00926,0.00501
30,close,0.00284,-0.0124,0.0181,0.00282,0.0078
60,close,-0.00155,-0.0239,0.0205,-0.00153,0.0114


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0517,0.043,0.0604,0.0517,0.00443
5,close,0.0207,0.0111,0.0302,0.0207,0.00485
15,close,0.0108,0.00116,0.0203,0.0108,0.00492
30,close,0.00476,-0.00994,0.0192,0.00478,0.00742
60,close,-0.00154,-0.0227,0.0195,-0.00148,0.0107


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,91.86%,91.45%,92.23%,91.84%,0.20%
5,close,64.41%,63.10%,65.64%,64.36%,0.65%
15,close,-0.34%,-2.56%,1.68%,-0.42%,1.09%
30,close,-1.24%,-3.40%,0.65%,-1.37%,1.03%
60,close,0.12%,-2.12%,2.24%,0.04%,1.11%


## VAR

In [16]:
RUN = False

var_prediction_path = (
    BASELINE_CACHE_ROOT
    / "var"
    / "prediction_result.pt"
)

if RUN:
    var = VarBaseline.from_config(
        config,
        maxlags=15,
        ic="aic",
        trend="c",
    )

    var.fit(
        train_split=train,
        val_split=val,
    )

    var_result = var.predict(
        split=test,
        batch_size=256,
    )

    var_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        var_result,
        var_prediction_path,
    )

else:
    var_result = torch.load(
        var_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_format = (
                "{:.2%}"
                if metric_name
                in ["cumulative_log_change_directional_accuracy",
                    "raw_price_temporal_pearson_correlation",
                    "forecast_series_log_return_temporal_pearson_correlation",
                    "persistence_win_rate",
                    "cumulative_log_change_movement_magnitude_ratio",
                    ]
                else "{:.3g}"
            )
    metric_display = (
        var_metric_table
        .loc[
            var_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000379,0.000364,0.000395,0.000379,7.87e-06
5,close,0.000799,0.00077,0.00083,0.000799,1.51e-05
15,close,0.00134,0.00129,0.00139,0.00134,2.49e-05
30,close,0.00185,0.00178,0.00192,0.00185,3.81e-05
60,close,0.00256,0.00244,0.00271,0.00256,6.97e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000265,—,—,—,—
5,close,0.000587,—,—,—,—
15,close,0.000985,—,—,—,—
30,close,0.00137,—,—,—,—
60,close,0.00191,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00111,—,—,—,—
5,close,0.00224,—,—,—,—
15,close,0.00373,—,—,—,—
30,close,0.00514,—,—,—,—
60,close,0.00712,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0109,-0.00487,0.0265,0.011,0.00802
5,close,0.0158,0.00093,0.0313,0.0159,0.0077
15,close,-0.00254,-0.0182,0.0127,-0.00249,0.00785
30,close,0.00438,-0.0124,0.0209,0.00445,0.00857
60,close,0.00226,-0.0126,0.0172,0.00232,0.00758


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.80%,99.85%,99.82%,0.01%
60,close,99.68%,99.60%,99.71%,99.66%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.04,1.03,1.04,1.04,0.00268
5,close,1.02,1.01,1.02,1.02,0.00231
15,close,1.01,1.01,1.01,1.01,0.00156
30,close,1,1,1.01,1,0.00125
60,close,1,1,1.01,1,0.000971


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,43.45%,42.95%,43.95%,43.46%,0.26%
5,close,46.98%,46.50%,47.45%,46.98%,0.24%
15,close,47.59%,47.08%,48.09%,47.59%,0.26%
30,close,48.28%,47.73%,48.83%,48.28%,0.28%
60,close,48.50%,47.76%,49.22%,48.49%,0.37%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.027,0.0178,0.0367,0.027,0.00483
5,close,0.00781,-0.00128,0.0171,0.00784,0.00472
15,close,-0.00775,-0.017,0.00179,-0.00778,0.00475
30,close,-0.00246,-0.0129,0.00827,-0.00249,0.00544
60,close,-0.00141,-0.015,0.0128,-0.00141,0.00715


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0275,0.0195,0.0353,0.0274,0.00405
5,close,0.0103,0.00299,0.0178,0.0103,0.00374
15,close,-0.000774,-0.00939,0.00803,-0.000768,0.00448
30,close,0.000408,-0.00858,0.00964,0.000415,0.00464
60,close,0.000339,-0.0123,0.0135,0.000358,0.00657


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,91.58%,91.16%,91.96%,91.56%,0.20%
5,close,63.86%,62.54%,65.08%,63.80%,0.65%
15,close,-0.34%,-2.60%,1.73%,-0.42%,1.11%
30,close,-1.22%,-3.34%,0.61%,-1.34%,1.01%
60,close,0.20%,-2.04%,2.33%,0.11%,1.12%


## GARCH

In [17]:
RUN = False

garch_prediction_path = (
    BASELINE_CACHE_ROOT
    / "garch"
    / "prediction_result.pt"
)

if RUN:
    garch = GarchBaseline.from_config(
        config,
        mean="AR",
        return_scale=10000.0,
    )

    garch.fit(
        train_split=train,
        val_split=val,
    )

    garch_result = garch.predict(
        split=test,
        batch_size=256,
    )

    garch_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        garch_result,
        garch_prediction_path,
    )

else:
    garch_result = torch.load(
        garch_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_display = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )
    metric_format = (
                "{:.2%}"
                if metric_name
                in ["cumulative_log_change_directional_accuracy",
                    "raw_price_temporal_pearson_correlation",
                    "forecast_series_log_return_temporal_pearson_correlation",
                    "persistence_win_rate",
                    "cumulative_log_change_movement_magnitude_ratio",
                    ]
                else "{:.3g}"
            )

    display(
        metric_display.style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000368,0.000353,0.000384,0.000368,7.91e-06
5,close,0.000785,0.000757,0.000816,0.000785,1.49e-05
15,close,0.00132,0.00128,0.00137,0.00132,2.45e-05
30,close,0.00184,0.00177,0.00192,0.00184,3.81e-05
60,close,0.00256,0.00243,0.00271,0.00256,7.02e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000255,—,—,—,—
5,close,0.000574,—,—,—,—
15,close,0.000978,—,—,—,—
30,close,0.00136,—,—,—,—
60,close,0.00191,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00109,—,—,—,—
5,close,0.0022,—,—,—,—
15,close,0.0037,—,—,—,—
30,close,0.00512,—,—,—,—
60,close,0.00711,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0319,0.00574,0.0551,0.0319,0.0125
5,close,0.0144,-0.00213,0.031,0.0143,0.00847
15,close,0.000941,-0.0153,0.0156,0.000997,0.00781
30,close,0.00278,-0.0101,0.015,0.00286,0.00642
60,close,0.00341,-0.0104,0.0176,0.00357,0.00714


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.80%,99.85%,99.83%,0.01%
60,close,99.68%,99.60%,99.72%,99.67%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.01,1,1.01,1.01,0.0013
5,close,0.999,0.998,1,0.999,0.000466
15,close,1,1,1,1,0.000427
30,close,1,0.999,1,1,0.000706
60,close,1,0.999,1,1,0.00123


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,45.69%,45.13%,46.27%,45.69%,0.29%
5,close,48.56%,47.88%,49.24%,48.56%,0.34%
15,close,48.75%,47.85%,49.65%,48.75%,0.46%
30,close,48.77%,47.53%,50.02%,48.77%,0.64%
60,close,48.21%,46.42%,49.97%,48.20%,0.91%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0456,0.0366,0.0548,0.0456,0.00465
5,close,0.0196,0.0113,0.0278,0.0195,0.00423
15,close,0.0135,0.00512,0.0216,0.0135,0.00421
30,close,0.0105,0.000427,0.0206,0.0105,0.00519
60,close,0.00884,-0.00551,0.0234,0.00892,0.00737


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0499,0.0415,0.0585,0.05,0.00426
5,close,0.0214,0.0137,0.029,0.0214,0.00392
15,close,0.0173,0.0107,0.024,0.0173,0.00337
30,close,0.0168,0.00788,0.026,0.0168,0.00461
60,close,0.0152,0.0024,0.0282,0.0152,0.00661


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,91.86%,91.45%,92.23%,91.84%,0.20%
5,close,64.42%,63.11%,65.65%,64.37%,0.65%
15,close,-0.34%,-2.56%,1.68%,-0.42%,1.09%
30,close,-1.24%,-3.40%,0.64%,-1.37%,1.03%
60,close,0.12%,-2.12%,2.24%,0.04%,1.11%


## ModernTCN — joint OHLCV baseline trained with cumulative-log-change MAE

This cell reports the cross-asset ModernTCN baseline selected for the dissertation table. Set `MODERN_TCN_RUN_DIR` to any completed standalone ModernTCN run directory containing `best_checkpoint.pt`.

The checkpoint is now authoritative: its saved input channels, target channels, joint/per-asset layout, patch geometry, hidden dimension, kernels, RevIN/context-normalisation settings and temporal-position setting are restored automatically. The current `forecasting.yaml` may describe a different ModernTCN run without blocking inference.

In [18]:
# Inputs:
# - RUN=True: load the selected checkpoint, generate test predictions, and
#   overwrite BASELINE_CACHE_ROOT/modern_tcn/prediction_result.pt.
# - RUN=False: load that existing prediction file without running the model.
# - MODERN_TCN_RUN_DIR: any completed ModernTCN run folder containing
#   best_checkpoint.pt. No match to forecasting.yaml is required.

RUN = False

MODERN_TCN_RUN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/checkpoints/modern_tcn/"
    "modern_tcn_joint_ohlcv_clgmae_d32_k1_p4s2_lk51_fs1"
).expanduser().resolve()

modern_tcn_prediction_path = (
    BASELINE_CACHE_ROOT
    / "modern_tcn"
    / "prediction_result.pt"
)

if RUN:
    modern_tcn_checkpoint_path = (
        MODERN_TCN_RUN_DIR
        / "best_checkpoint.pt"
    )

    modern_tcn = ModernTCNBaseline.from_checkpoint(
        checkpoint_path=modern_tcn_checkpoint_path,
        device="cpu",
    )

    modern_tcn_result = modern_tcn.predict(
        split=test,
        batch_size=8,
        num_workers=0,
    )
    modern_tcn_result["asset_cols"] = list(test["asset_cols"])
    modern_tcn_result["output_space"] = "raw"

    modern_tcn_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    torch.save(
        modern_tcn_result,
        modern_tcn_prediction_path,
    )
else:
    modern_tcn_result = torch.load(
        modern_tcn_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

modern_tcn_evaluator = ForecastEvaluator(
    prediction_result=modern_tcn_result,
    train_split=train,
)
modern_tcn_results = modern_tcn_evaluator.evaluate(
    metrics=modern_tcn_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)
modern_tcn_metric_table = make_evaluation_table(
    metric_results=modern_tcn_results,
    horizons=modern_tcn_evaluator.horizons,
    channels=modern_tcn_evaluator.channels,
)

for metric_name in modern_tcn_evaluator.available_metrics:
    metric_format = (
                "{:.2%}"
                if metric_name
                in ["cumulative_log_change_directional_accuracy",
                    "raw_price_temporal_pearson_correlation",
                    "forecast_series_log_return_temporal_pearson_correlation",
                    "persistence_win_rate",
                    "cumulative_log_change_movement_magnitude_ratio",
                    ]
                else "{:.3g}"
            )
    display(
        modern_tcn_metric_table
        .loc[modern_tcn_metric_table["metric"] == metric_name]
        .set_index(["horizon", "channel"])[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000374,0.00036,0.00039,0.000374,7.79e-06
5,close,0.000796,0.000767,0.000826,0.000796,1.52e-05
15,close,0.00133,0.00128,0.00138,0.00133,2.45e-05
30,close,0.00185,0.00177,0.00192,0.00185,3.87e-05
60,close,0.00257,0.00244,0.00271,0.00257,7.06e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000262,—,—,—,—
5,close,0.00058,—,—,—,—
15,close,0.000981,—,—,—,—
30,close,0.00136,—,—,—,—
60,close,0.00191,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00111,—,—,—,—
5,close,0.00224,—,—,—,—
15,close,0.00371,—,—,—,—
30,close,0.00515,—,—,—,—
60,close,0.00713,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0138,-0.00763,0.0344,0.0137,0.0108
5,close,0.0105,-0.00676,0.0279,0.0104,0.0088
15,close,0.0142,-0.00626,0.0335,0.0143,0.0101
30,close,0.0133,-0.0104,0.0383,0.0136,0.0125
60,close,0.00657,-0.0245,0.0357,0.00711,0.0155


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.80%,99.85%,99.82%,0.01%
60,close,99.67%,99.60%,99.71%,99.66%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.03,1.02,1.03,1.03,0.00196
5,close,1.01,1.01,1.02,1.01,0.00213
15,close,1.01,1,1.01,1.01,0.00125
30,close,1,1,1.01,1,0.00145
60,close,1,1,1.01,1,0.0018


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,44.22%,43.63%,44.82%,44.22%,0.30%
5,close,47.25%,46.57%,47.96%,47.25%,0.35%
15,close,48.33%,47.77%,48.90%,48.33%,0.29%
30,close,48.57%,47.88%,49.24%,48.56%,0.34%
60,close,48.22%,47.30%,49.08%,48.22%,0.46%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0172,0.00745,0.0266,0.0172,0.0049
5,close,0.0117,-0.000121,0.0237,0.0117,0.00607
15,close,0.0134,0.00225,0.0246,0.0134,0.00571
30,close,0.0209,0.00633,0.0348,0.0208,0.00723
60,close,0.0218,0.00404,0.0381,0.0217,0.00863


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0229,0.0161,0.0299,0.0229,0.00351
5,close,0.0174,0.00848,0.0265,0.0173,0.00459
15,close,0.0175,0.00835,0.0264,0.0174,0.00457
30,close,0.0235,0.0115,0.0348,0.0234,0.00598
60,close,0.0193,0.00457,0.0332,0.0192,0.00719


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,91.73%,91.31%,92.09%,91.70%,0.20%
5,close,63.88%,62.51%,65.14%,63.82%,0.67%
15,close,-0.14%,-2.40%,1.93%,-0.22%,1.11%
30,close,-1.07%,-3.25%,0.88%,-1.20%,1.06%
60,close,0.27%,-2.05%,2.45%,0.18%,1.15%


## Final continuous dynamic-graph model

This cell evaluates a completed continuous-forecaster run directory on the held-out test split. The run folder is the source of truth: `resolved_config.json` reconstructs the exact temporal backbone, graph learner, spatial mixer, learned beta gate and output representation, while `best_checkpoint.pt` supplies the selected weights.

With `RUN=True`, the helper performs one chronological test inference pass and writes the following files beside `best_checkpoint.pt`:

- `test_predictions.pt`
- `test_graphs.pt`
- `test_metric_table.csv`
- `test_diagnostics.json`

With `RUN=False`, it reloads `test_predictions.pt` and recomputes the common metrics and bootstrap table without running the model again.

In [19]:
FINAL_MODEL_RUN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/models_to_use/final_model"
).expanduser().resolve()

final_model_artifacts = load_evaluation_artifacts(
    FINAL_MODEL_RUN_DIR,
    split="test",
    policy="best",
)

final_model_result = (
    final_model_artifacts.prediction_result
)

final_model_evaluator = ForecastEvaluator(
    prediction_result=final_model_result,
    train_split=train,
)

final_model_results = final_model_evaluator.evaluate(
    metrics=final_model_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

final_model_metric_table = make_evaluation_table(
    metric_results=final_model_results,
    horizons=final_model_evaluator.horizons,
    channels=final_model_evaluator.channels,
)

for metric_name in final_model_results:
    metric_format = (
        "{:.2%}"
        if metric_name
        in ["cumulative_log_change_directional_accuracy",
            "raw_price_temporal_pearson_correlation",
            "forecast_series_log_return_temporal_pearson_correlation",
            "persistence_win_rate",
            "cumulative_log_change_movement_magnitude_ratio",
            ]
        else "{:.3g}"
    )

    display(
        final_model_metric_table
        .loc[
            final_model_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000368,0.000354,0.000384,0.000368,7.81e-06
5,close,0.000786,0.000758,0.000816,0.000786,1.49e-05
15,close,0.00132,0.00128,0.00137,0.00132,2.43e-05
30,close,0.00184,0.00177,0.00192,0.00184,3.8e-05
60,close,0.00256,0.00243,0.0027,0.00256,7.02e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000257,—,—,—,—
5,close,0.000575,—,—,—,—
15,close,0.000979,—,—,—,—
30,close,0.00136,—,—,—,—
60,close,0.0019,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0011,—,—,—,—
5,close,0.00221,—,—,—,—
15,close,0.00369,—,—,—,—
30,close,0.00512,—,—,—,—
60,close,0.00712,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0496,0.0285,0.071,0.0494,0.0109
5,close,0.0219,0.00773,0.0362,0.0219,0.00736
15,close,0.0372,0.0134,0.0614,0.0372,0.0123
30,close,0.0166,-0.00584,0.0388,0.0165,0.0114
60,close,0.00759,-0.0338,0.0443,0.00822,0.0203


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.80%,99.85%,99.83%,0.01%
60,close,99.68%,99.60%,99.72%,99.66%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1,1,1.01,1,0.00104
5,close,1,0.999,1,1,0.000628
15,close,1,0.999,1,1,0.000757
30,close,1,1,1,1,0.000711
60,close,1,0.997,1,1,0.00133


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,45.56%,44.90%,46.23%,45.56%,0.34%
5,close,48.77%,48.20%,49.33%,48.77%,0.29%
15,close,49.26%,48.40%,50.10%,49.26%,0.43%
30,close,49.28%,48.57%,49.97%,49.27%,0.36%
60,close,49.24%,48.19%,50.23%,49.23%,0.52%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0383,0.026,0.0501,0.0383,0.00611
5,close,0.0192,0.0097,0.029,0.0193,0.0049
15,close,0.0143,0.00262,0.0258,0.0143,0.00589
30,close,0.0131,-0.00132,0.0269,0.013,0.00726
60,close,0.0254,0.00921,0.0411,0.0252,0.00811


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0393,0.0305,0.0477,0.0393,0.00437
5,close,0.0225,0.0141,0.031,0.0226,0.00429
15,close,0.0126,0.0033,0.0219,0.0126,0.00476
30,close,0.0135,0.00253,0.0242,0.0134,0.00553
60,close,0.0235,0.00978,0.0361,0.0233,0.00665


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,91.88%,91.46%,92.25%,91.85%,0.20%
5,close,64.34%,63.01%,65.57%,64.28%,0.65%
15,close,-0.16%,-2.36%,1.85%,-0.24%,1.08%
30,close,-1.20%,-3.38%,0.70%,-1.33%,1.04%
60,close,0.16%,-2.10%,2.28%,0.07%,1.11%


## Kronos

In [20]:
kronos_result = torch.load(
    Path(
        "/Users/vishalruparelia/Library/CloudStorage/"
        "GoogleDrive-vishal@autonomous-fox.ai/"
        "My Drive/dissertation/kronos/"
        "kronos_small_test_fp16_bs16_progress.pt"
    ),
    map_location="cpu",
    weights_only=False,
)["prediction_result"]

kronos_evaluator = ForecastEvaluator(
    prediction_result=kronos_result,
    train_split=train,
)

kronos_results = kronos_evaluator.evaluate(
    metrics=kronos_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

kronos_metric_table = make_evaluation_table(
    metric_results=kronos_results,
    horizons=kronos_evaluator.horizons,
    channels=kronos_evaluator.channels,
)

for metric_name in kronos_evaluator.available_metrics:
    metric_format = (
            "{:.2%}"
            if metric_name
            in ["cumulative_log_change_directional_accuracy",
                "raw_price_temporal_pearson_correlation",
                "forecast_series_log_return_temporal_pearson_correlation",
                "persistence_win_rate",
                "cumulative_log_change_movement_magnitude_ratio",
                ]
            else "{:.3g}"
        )
    display(
        kronos_metric_table
        .loc[kronos_metric_table["metric"] == metric_name]
        .set_index(["horizon", "channel"])[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format(metric_format,na_rep="—",)
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000409,0.000393,0.000426,0.000409,8.41e-06
5,close,0.000824,0.000793,0.000856,0.000824,1.61e-05
15,close,0.00137,0.00132,0.00142,0.00137,2.6e-05
30,close,0.0019,0.00182,0.00198,0.0019,4.01e-05
60,close,0.00264,0.00251,0.0028,0.00265,7.47e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000278,—,—,—,—
5,close,0.000595,—,—,—,—
15,close,0.001,—,—,—,—
30,close,0.0014,—,—,—,—
60,close,0.00195,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00121,—,—,—,—
5,close,0.00233,—,—,—,—
15,close,0.00384,—,—,—,—
30,close,0.0053,—,—,—,—
60,close,0.00738,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00216,-0.0473,0.0385,0.00257,0.0224
5,close,0.0106,-0.0101,0.0305,0.0106,0.0104
15,close,-0.0158,-0.0341,0.00164,-0.0155,0.0093
30,close,-0.0173,-0.0336,0.00016,-0.017,0.00868
60,close,-0.0268,-0.066,0.00769,-0.0259,0.0192


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.96%,99.96%,99.97%,99.96%,0.00%
15,close,99.90%,99.88%,99.91%,99.90%,0.01%
30,close,99.82%,99.78%,99.84%,99.81%,0.01%
60,close,99.65%,99.57%,99.70%,99.64%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.13,1.11,1.14,1.13,0.0072
5,close,1.05,1.04,1.06,1.05,0.00385
15,close,1.04,1.03,1.04,1.04,0.00258
30,close,1.03,1.03,1.04,1.03,0.00255
60,close,1.03,1.03,1.04,1.03,0.00326


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,42.31%,41.83%,42.81%,42.31%,0.25%
5,close,46.00%,45.64%,46.38%,46.00%,0.19%
15,close,46.06%,45.73%,46.42%,46.07%,0.18%
30,close,46.07%,45.64%,46.49%,46.07%,0.22%
60,close,45.99%,45.41%,46.54%,45.98%,0.29%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0322,0.0227,0.0417,0.0322,0.00486
5,close,0.0165,0.00511,0.0277,0.0165,0.00573
15,close,0.00868,-0.0019,0.0196,0.00874,0.00545
30,close,0.00897,-0.00147,0.0192,0.00894,0.00533
60,close,0.0107,-0.00418,0.0247,0.0106,0.00739


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0387,0.0307,0.0468,0.0386,0.0041
5,close,0.021,0.0133,0.0288,0.021,0.00392
15,close,0.00752,-0.000454,0.0155,0.00754,0.00405
30,close,0.00505,-0.00395,0.0138,0.00501,0.00451
60,close,0.00638,-0.00566,0.0179,0.00628,0.00599


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,88.56%,87.99%,89.17%,88.59%,0.30%
5,close,60.12%,58.60%,61.46%,60.06%,0.73%
15,close,-0.32%,-2.43%,1.53%,-0.41%,1.02%
30,close,-1.22%,-3.36%,0.67%,-1.35%,1.04%
60,close,-0.05%,-2.14%,1.95%,-0.13%,1.05%


## Token GraphTCN

In [21]:
# Token GraphTCN
#
# Inputs:
# - FINAL_MODEL_TOKENS_RUN_DIR: frozen Token GraphTCN run folder.
# - TOKEN_POLICY=None: automatically use the sampling policy selected in the
#   run's saved temperature-selection manifest. Set an explicit value such as
#   "temperature_1" only when you deliberately want that policy.

FINAL_MODEL_TOKENS_RUN_DIR = (
    BASELINE_CACHE_ROOT
    / "models_to_use"
    / "final_model_tokens_post"
).expanduser().resolve()

TOKEN_POLICY = None

final_model_tokens_artifacts = load_evaluation_artifacts(
    FINAL_MODEL_TOKENS_RUN_DIR,
    split="test",
    policy=TOKEN_POLICY,
)

final_model_tokens_result = (
    final_model_tokens_artifacts.prediction_result
)


final_model_tokens_evaluator = ForecastEvaluator(
    prediction_result=final_model_tokens_result,
    train_split=train,
)

final_model_tokens_results = (
    final_model_tokens_evaluator.evaluate(
        metrics=final_model_tokens_evaluator.available_metrics,
        reduce_dims=(0, 2),
        bootstrap=True,
        n_bootstrap=BOOTSTRAP_N,
        confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
        bootstrap_seed=BOOTSTRAP_SEED,
    )
)

final_model_tokens_metric_table = make_evaluation_table(
    metric_results=final_model_tokens_results,
    horizons=final_model_tokens_evaluator.horizons,
    channels=final_model_tokens_evaluator.channels,
)

percentage_metrics = {
    "cumulative_log_change_directional_accuracy",
    "raw_price_temporal_pearson_correlation",
    "forecast_series_log_return_temporal_pearson_correlation",
    "persistence_win_rate",
    "cumulative_log_change_movement_magnitude_ratio",
}

for metric_name in final_model_tokens_evaluator.available_metrics:
    metric_format = (
        "{:.2%}"
        if metric_name in percentage_metrics
        else "{:.3g}"
    )

    display(
        final_model_tokens_metric_table
        .loc[
            final_model_tokens_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format(
            metric_format,
            na_rep="—",
        )
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000401,0.000386,0.000417,0.000401,8.12e-06
5,close,0.000805,0.000776,0.000837,0.000805,1.54e-05
15,close,0.00134,0.00129,0.00139,0.00134,2.47e-05
30,close,0.00185,0.00178,0.00193,0.00185,3.85e-05
60,close,0.00257,0.00244,0.00272,0.00257,7.12e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00028,—,—,—,—
5,close,0.000588,—,—,—,—
15,close,0.00099,—,—,—,—
30,close,0.00137,—,—,—,—
60,close,0.0019,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00118,—,—,—,—
5,close,0.00226,—,—,—,—
15,close,0.00374,—,—,—,—
30,close,0.00516,—,—,—,—
60,close,0.00714,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00699,-0.0296,0.0365,0.00729,0.0171
5,close,0.0113,-0.00353,0.0266,0.0112,0.00775
15,close,-9.7e-05,-0.0261,0.024,9.16e-05,0.0129
30,close,-0.000993,-0.0214,0.0186,-0.000957,0.0103
60,close,-0.0128,-0.0417,0.0152,-0.0123,0.0147


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,99.99%,99.99%,99.99%,99.99%,0.00%
5,close,99.97%,99.96%,99.97%,99.97%,0.00%
15,close,99.91%,99.89%,99.92%,99.91%,0.01%
30,close,99.83%,99.79%,99.85%,99.82%,0.01%
60,close,99.67%,99.60%,99.71%,99.66%,0.03%


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.1,1.09,1.11,1.1,0.00456
5,close,1.02,1.02,1.03,1.02,0.00256
15,close,1.01,1.01,1.01,1.01,0.00185
30,close,1.01,1,1.01,1.01,0.00148
60,close,1.01,1,1.01,1.01,0.00172


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,41.59%,41.18%,42.02%,41.59%,0.22%
5,close,46.34%,45.98%,46.70%,46.34%,0.18%
15,close,47.47%,46.97%,47.97%,47.48%,0.26%
30,close,47.85%,47.20%,48.51%,47.86%,0.34%
60,close,47.77%,46.72%,48.80%,47.77%,0.53%


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.021,0.0103,0.032,0.021,0.0055
5,close,0.0151,0.00583,0.0242,0.0151,0.00467
15,close,0.0146,0.00464,0.0244,0.0146,0.00506
30,close,0.0125,0.00265,0.0222,0.0125,0.00493
60,close,0.0122,0.00099,0.0234,0.0121,0.00566


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0288,0.0205,0.0373,0.0288,0.00431
5,close,0.0186,0.0114,0.0256,0.0185,0.00359
15,close,0.0152,0.00822,0.0223,0.0152,0.0036
30,close,0.0131,0.00542,0.0208,0.013,0.0039
60,close,0.013,0.00414,0.0216,0.013,0.00445


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,90.50%,90.09%,90.92%,90.50%,0.21%
5,close,62.54%,61.14%,63.83%,62.48%,0.69%
15,close,-0.22%,-2.40%,1.78%,-0.30%,1.07%
30,close,-1.33%,-3.60%,0.62%,-1.46%,1.08%
60,close,0.00%,-2.22%,2.05%,-0.08%,1.09%
